In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report


In [ ]:
raw_file = 'training.csv'

In [ ]:
raw_df = pd.read_csv(raw_file)

In [ ]:
raw_df.head()

In [ ]:
raw_df.shape

In [ ]:
raw_df.describe()

In [ ]:
print(raw_df['Label'].value_counts()) 

In [ ]:
print("\nTarget distribution:")
print(raw_df['Label'].value_counts())
print(raw_df['Label'].value_counts(normalize=True))

In [ ]:
raw_df = raw_df.replace(-999.0, np.nan)
print("\nMissing values per column (before handling):")
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0].sort_values(ascending=False))

In [ ]:
missing_cols = [
    'DER_deltaeta_jet_jet',
    'DER_mass_jet_jet',
    'DER_prodeta_jet_jet',
    'PRI_jet_subleading_phi',
    'DER_lep_eta_centrality',
    'PRI_jet_subleading_eta',
    'PRI_jet_subleading_pt',
    'PRI_jet_leading_eta',
    'PRI_jet_leading_pt',
    'PRI_jet_leading_phi',
    'DER_mass_MMC'
]


In [ ]:
for col in missing_cols:
    raw_df[f'{col}_is_missing'] = raw_df[col].isnull().astype(int)

In [ ]:
raw_df

In [ ]:
raw_df = raw_df.fillna(-999)
print("\nMissing values after handling:")
print(raw_df.isnull().sum().sum())

In [ ]:
X = raw_df.drop(['EventId', 'Label', 'Weight'], axis=1)
y = raw_df['Label'].map({'s': 1, 'b': 0})
weights = raw_df['Weight']

In [ ]:
print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Weight summary: min={weights.min():.2f}, max={weights.max():.2f}, mean={weights.mean():.2f}")

In [ ]:
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X, y, weights, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
print(f"\nTrain size: {len(X_train)}, Validation size: {len(X_val)}")

In [ ]:
modelA = RandomForestClassifier(n_jobs=-1, random_state=42).fit(X_train, y_train)

In [ ]:
y_predA = modelA.predict(X_val)
modelA.score(X_val, y_val)

In [ ]:
modelB = RandomForestClassifier(n_jobs=-1, random_state=42).fit(X_train, y_train, sample_weight=w_train)
y_predB = modelB.predict(X_val)

In [ ]:
accuracy_score(y_val, y_predB)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_val, y_predA))  # Model A
print(classification_report(y_val, y_predB))  # Model B

In [ ]:
def ams_score(y_true, y_pred, br=10):
    s = sum((y_true == 1) & (y_pred == 1))
    b = sum((y_true == 0) & (y_pred == 1))
    return np.sqrt(2 * ((s + b + br) * np.log(1 + s/(b + br)) - s))

print(ams_score(y_val, y_predA))  # Model A
print(ams_score(y_val, y_predB))  # Model B

In [ ]:
def test_params(**params):
    model_alpha = RandomForestClassifier(n_jobs=-1, random_state=42, **params).fit(X_train, y_train, sample_weight=w_train)
    y_pred_alpha = model_alpha.predict(X_val)
    print(ams_score(y_val, y_pred_alpha))
    print(ams_score(y_val, y_predA))
    print(ams_score(y_val, y_predB))

In [ ]:
test_params(max_depth=100)

In [ ]:
test_params(max_depth=50)

In [ ]:
test_params(max_leaf_nodes=2**20)

In [ ]:
%%time
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Smaller parameter grid (8 combinations × 2 folds = 16 fits)
param_grid_small = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15],
    'min_samples_split': [5]
}

# Use 2-fold CV instead of 3
grid_small = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_small,
    cv=2,
    scoring=ams_score,
    n_jobs=-1,
    verbose=1
)

# Fit with weights (or without — your choice)
grid_small.fit(X_train, y_train, sample_weight=w_train)

In [ ]:
# Final model with best parameters
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)

# Train on full training set (with weights)
final_model.fit(X_train, y_train, sample_weight=w_train)

# Evaluate on validation set
y_pred = final_model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")

ams_val = ams_score(y_val, y_pred)
print(f"Validation AMS: {ams_val:.4f}")

In [ ]:
final_model_no_weight = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
final_model_no_weight.fit(X_train, y_train)  # no sample_weight

y_pred_no_weight = final_model_no_weight.predict(X_val)
print(f"Accuracy (no weights): {accuracy_score(y_val, y_pred_no_weight):.4f}")
print(f"AMS (no weights): {ams_score(y_val, y_pred_no_weight):.4f}")

In [ ]:
param_grid_small_157 = {
    'n_estimators': [100, 200],
    'min_samples_split': [2, 5],
    'max_depth': [None, 20],
    'max_features': ['sqrt', 0.5]
}
# 2 × 2 × 2 × 2 = 16 combinations × 2 folds = 32 fits → ~20 min

In [ ]:
%%time
grid_157_small = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_small_157,
    cv=2,
    scoring=ams_score,
    n_jobs=-1,
    verbose=1
)

grid_157_small.fit(X_train, y_train)

In [ ]:
# This is the best model from GridSearchCV
final_model = grid_157_small.best_estimator_

# Evaluate on validation set one last time
y_pred = final_model.predict(X_val)
ams_val = ams_score(y_val, y_pred)
print(f"Validation AMS: {ams_val}")

In [ ]:
print(':)')
print(':)')
print(':)')

In [ ]:
import pandas as pd
import numpy as np

# Load test data
test_df = pd.read_csv('test.csv')

# Apply same preprocessing
test_df = test_df.replace(-999.0, np.nan)
for col in missing_cols:
    test_df[f'{col}_is_missing'] = test_df[col].isnull().astype(int)
test_df = test_df.fillna(-999)

# Get probabilities (not just predictions)
X_test = test_df.drop(['EventId'], axis=1)
y_proba = final_model.predict_proba(X_test)[:, 1]  # Probability of being signal (class 1)

# Create submission DataFrame
submission = pd.DataFrame({
    'EventId': test_df['EventId'],
    'Probability': y_proba
})

# Rank by probability (highest probability = rank 1)
submission['RankOrder'] = submission['Probability'].rank(ascending=False, method='first').astype(int)

# Map probability to class (threshold 0.5)
submission['Class'] = np.where(submission['Probability'] >= 0.5, 's', 'b')

# Keep only required columns
submission = submission[['EventId', 'RankOrder', 'Class']]

# Save
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv created!")
print(submission.head(10))